# Preference Tuning of a Causal LLM with DPO (Hugging Face + TRL)

This notebook focuses **only on preference tuning** using Direct Preference Optimization (DPO). It takes a base (or already instruction-tuned) causal language model and trains it on `prompt` / `chosen` / `rejected` preference data, so the model learns to prefer better responses over weaker ones.

No non-instruction (raw text) pretraining and no instruction fine-tuning steps are included here — this notebook is self-contained and starts directly from a base pretrained model. If you already have an instruction-tuned model, you can simply point `config.model_name` at it instead.


## What is Preference Tuning (DPO)?

In preference tuning, training data consists of a prompt plus two candidate responses:

```json
{
  "prompt": "Explain the mechanism of action of Metformin.",
  "chosen": "Metformin primarily activates AMPK, which improves glucose uptake and reduces hepatic gluconeogenesis.",
  "rejected": "Metformin lowers blood sugar somehow."
}
```

- `prompt` is the user instruction.
- `chosen` is the preferred/better answer.
- `rejected` is the weaker answer.

**Direct Preference Optimization (DPO)** trains the model directly on this preference data, without needing a separate reward model or an RL loop like PPO. The model learns to assign a higher probability/reward to the `chosen` response and a lower one to the `rejected` response, while staying close to a reference model (by default, a frozen copy of the starting model).


## Instruction Fine-Tuning vs DPO

| Aspect | Instruction Fine-Tuning (SFT) | DPO (Preference Tuning) |
|---|---|---|
| Data format | `instruction` / `input` / `output` | `prompt` / `chosen` / `rejected` |
| What model learns | How to answer at all | Which of two answers is better |
| Reward model needed? | No | No (DPO removes this need vs. classic RLHF) |
| RL loop (PPO) needed? | No | No |
| Typical use | Teach task-following behavior | Refine/align behavior, safety, style, quality |


## Pipeline

```text
Preference dataset (JSONL: prompt / chosen / rejected)
   ↓
Train/validation split
   ↓
Load base model (4-bit QLoRA if GPU available)
   ↓
Attach a fresh LoRA adapter
   ↓
Configure DPO (DPOConfig: beta, max_length, max_prompt_length, ...)
   ↓
Train with DPOTrainer (TRL)
   ↓
Save preference-tuned LoRA adapter
   ↓
Reload adapter for inference
   ↓
Test preference-tuned responses
   ↓
(Optional) Merge adapter into base model and save standalone model
```


In [1]:
!pip install -q -U datasets transformers accelerate peft bitsandbytes>=0.46.1 "trl>=0.16.0" "torchao>=0.16.0"

In [2]:
import os
import re
import gc
import math
import json
import random
import unicodedata
import torch
import inspect
from dataclasses import dataclass, asdict
from typing import List, Dict, Any
from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, set_seed,
)
from peft import (
    LoraConfig, TaskType, get_peft_model,
    prepare_model_for_kbit_training, PeftModel,
)
from trl import DPOConfig, DPOTrainer
from google.colab import userdata
from huggingface_hub import HfApi

## 2. Global configuration

In [3]:
@dataclass
class Config:
    # Path to the preference dataset in JSONL format.
    # Each line should look like:
    # {"prompt": "...", "chosen": "...", "rejected": "..."}
    preference_data_path: str = "/content/pharma_preference_dataset.jsonl"

    # Base (or instruction-tuned) causal language model to preference-tune.
    # Point this at a local instruction-tuned model directory if you have one.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during DPO training.
    output_dir: str = "/content/pharma_tinyllama_preference_dpo_output"

    # Directory where the final trained DPO LoRA adapter will be saved.
    adapter_dir: str = "/content/pharma_tinyllama_preference_dpo_lora_adapter"

    # Directory where the final merged (adapter + base) model will be saved.
    merged_model_dir: str = "/content/pharma_tinyllama_preference_merged_model"

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 3.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    # NOTE: with small preference datasets, keep this <= your number of
    # training examples, otherwise the Trainer will silently need multiple
    # passes over the data to complete a single optimizer step, inflating
    # the effective number of epochs actually run.
    gradient_accumulation_steps: int = 4

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 5e-5

    # Number of warmup steps used to gradually increase the learning rate.
    warmup_steps: int = 2

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps: int = 1
    logging_first_step: bool = True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 1

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 10

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

    # DPO-specific hyperparameter: controls how strongly the model is
    # pushed toward chosen answers over rejected answers (vs. the
    # reference model). Common values: 0.1 (default-ish), up to ~0.5.
    beta: float = 0.1

    # Maximum total sequence length (prompt + response) for DPO.
    max_length: int = 512

    # Maximum prompt length (truncated from the left if longer).
    max_prompt_length: int = 256


In [4]:
config = Config()
config

Config(preference_data_path='/content/pharma_preference_dataset.jsonl', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_preference_dpo_output', adapter_dir='/content/pharma_tinyllama_preference_dpo_lora_adapter', merged_model_dir='/content/pharma_tinyllama_preference_merged_model', test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=4, learning_rate=5e-05, warmup_steps=2, weight_decay=0.01, logging_steps=1, logging_first_step=True, eval_steps=1, save_steps=10, save_total_limit=2, max_steps=-1, beta=0.1, max_length=512, max_prompt_length=256)

In [5]:
print(json.dumps(asdict(config), indent=2))

{
  "preference_data_path": "/content/pharma_preference_dataset.jsonl",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_preference_dpo_output",
  "adapter_dir": "/content/pharma_tinyllama_preference_dpo_lora_adapter",
  "merged_model_dir": "/content/pharma_tinyllama_preference_merged_model",
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 4,
  "learning_rate": 5e-05,
  "warmup_steps": 2,
  "weight_decay": 0.01,
  "logging_steps": 1,
  "logging_first_step": true,
  "eval_steps": 1,
  "save_steps": 10,
  "save_total_limit": 2,
  "max_steps": -1,
  "beta": 0.1,
  "max_length": 512,
  "max_prompt_length": 256
}


In [6]:
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.merged_model_dir, exist_ok=True)

In [7]:
if not os.path.exists(config.preference_data_path):
    print(f"Preference dataset not found at: {config.preference_data_path}")
else:
    print(f"Preference dataset found: {config.preference_data_path}")

Preference dataset found: /content/pharma_preference_dataset.jsonl


## 3. Load the preference dataset

The dataset is expected to be a JSONL file where each line has `prompt`, `chosen`, and `rejected` fields. This is exactly the format `DPOTrainer` expects, so no extra formatting step is needed here (unlike instruction fine-tuning's Alpaca-style formatting).


In [8]:
preference_dataset = load_dataset(
    "json",
    data_files=config.preference_data_path,
    split="train"
)

print(preference_dataset)

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
    num_rows: 48
})


In [9]:
print(preference_dataset[0])

{'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:\n', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


## 4. Train/validation split

In [10]:
preference_dataset = preference_dataset.train_test_split(
    test_size=config.test_size,
    seed=config.seed
)

In [11]:
# Rename test split to validation split.
preference_dataset["validation"] = preference_dataset.pop("test")

In [12]:
print(preference_dataset)
print("Train rows:", len(preference_dataset["train"]))
print("Validation rows:", len(preference_dataset["validation"]))

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 8
    })
})
Train rows: 40
Validation rows: 8


## 5. Load tokenizer

The tokenizer converts text into token IDs. `DPOTrainer` uses it internally to tokenize `prompt`, `chosen`, and `rejected` fields.


In [13]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


## 6. Load base model for preference tuning (LoRA/QLoRA)

If a GPU is available, we load the model in 4-bit mode (QLoRA) to reduce memory usage. Otherwise, we fall back to full precision on CPU (slower).


In [14]:
use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")

CUDA available: True


In [15]:
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [16]:
if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    base_model = prepare_model_for_kbit_training(base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

base_model.config.use_cache = False

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 7. Attach a LoRA adapter

LoRA trains a small number of adapter parameters instead of the full model, which makes preference tuning much cheaper and faster.


In [17]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

preference_model = get_peft_model(base_model, lora_config)
preference_model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## 8. Configure DPO training

`DPOConfig` extends `TrainingArguments` with DPO-specific fields:

- `beta`: how strongly the model is pushed toward `chosen` over `rejected`, relative to the reference model.
- `max_length` / `max_prompt_length`: truncation limits for the full sequence and for the prompt portion.

Since `ref_model=None` is passed to `DPOTrainer` below, TRL will automatically use a frozen copy of the policy model's base weights as the reference model (this works cleanly with a LoRA adapter, since the adapter can be disabled internally to recover the reference behavior).


In [18]:
dpo_training_args = DPOConfig(
    output_dir=config.output_dir,

    # Training duration.
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,

    # Batch settings.
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,

    # Optimizer settings.
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,

    # Logging and evaluation.
    logging_steps=config.logging_steps,
    logging_first_step=config.logging_first_step,
    eval_strategy="steps",
    eval_steps=config.eval_steps,

    # Checkpoint saving.
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,

    # Precision settings.
    fp16=False,
    bf16=False,

    # Disable external logging tools.
    report_to="none",

    # Keep required columns.
    remove_unused_columns=False,

    # DPO hyperparameters.
    beta=config.beta,
)

In [20]:
# print(dpo_training_args)

## 9. Build DPOTrainer

In [21]:
dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,  # None means TRL will internally use the reference behavior
    args=dpo_training_args,

    train_dataset=preference_dataset["train"],
    eval_dataset=preference_dataset["validation"],

    processing_class=tokenizer,
)

print("DPOTrainer is ready.")

DPOTrainer is ready.


## 10. Start DPO Preference Tuning

In [22]:
dpo_train_result = dpo_trainer.train()

print("DPO preference tuning completed.")
print(dpo_train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.693147,0.693147,2.202081,679.000000,-3.535502,-3.529087,0.545947,0.000000,0.000000,0.000000,0.000000,-132.283320,-121.341744
2,0.693147,0.662175,2.200058,1413.000000,-3.537213,-3.531572,0.544479,0.025761,-0.038265,1.000000,0.064026,-132.025707,-121.724395
3,0.638374,0.587139,2.196331,2099.000000,-3.539134,-3.534807,0.541875,0.076082,-0.155655,1.000000,0.231737,-131.522503,-122.898296
4,0.560188,0.529790,2.193584,2789.000000,-3.540781,-3.537822,0.543925,0.121230,-0.256497,1.000000,0.377728,-131.071016,-123.906716
5,0.573051,0.459826,2.188666,3431.000000,-3.542232,-3.540823,0.543925,0.166600,-0.406239,1.000000,0.572839,-130.617320,-125.404133
6,0.482302,0.389207,2.183382,4051.000000,-3.543190,-3.543248,0.550472,0.206633,-0.593419,1.000000,0.800052,-130.216992,-127.275938
7,0.395303,0.328119,2.176794,4703.000000,-3.543714,-3.544987,0.550472,0.245434,-0.787746,1.000000,1.033180,-129.828982,-129.219208
8,0.345964,0.278875,2.170159,5380.000000,-3.544290,-3.546921,0.550472,0.279090,-0.974003,1.000000,1.253093,-129.492417,-131.081772
9,0.203065,0.237727,2.163554,6052.000000,-3.545051,-3.549440,0.551942,0.312747,-1.155499,1.000000,1.468246,-129.155852,-132.896734
10,0.264503,0.202426,2.157505,6698.000000,-3.545685,-3.552057,0.550319,0.341367,-1.336908,1.000000,1.678275,-128.869653,-134.710829


DPO preference tuning completed.
TrainOutput(global_step=30, training_loss=0.1941264892773082, metrics={'train_runtime': 193.6123, 'train_samples_per_second': 0.62, 'train_steps_per_second': 0.155, 'total_flos': 147834252288000.0, 'train_loss': 0.1941264892773082, 'epoch': 3.0})


### Understanding the DPO training logs

| Parameter               | Short Meaning                                                                              |
| ----------------------- | ------------------------------------------------------------------------------------------ |
| **Step**                | Current optimizer step during training.                                                    |
| **Training Loss**       | DPO loss on the training data; lower is generally better.                                  |
| **Validation Loss**     | DPO loss on unseen validation data; helps check generalization.                            |
| **Entropy**             | Measures how uncertain the model is; higher means more random, lower means more confident. |
| **Num Tokens**          | Total number of tokens processed so far.                                                   |
| **Logits/chosen**       | Raw model score for the preferred answer.                                                  |
| **Logits/rejected**     | Raw model score for the rejected answer.                                                   |
| **Mean Token Accuracy** | Average token-level prediction accuracy.                                                   |
| **Rewards/chosen**      | DPO implicit reward for the preferred answer; should be higher.                            |
| **Rewards/rejected**    | DPO implicit reward for the rejected answer; should be lower.                              |
| **Rewards/accuracies**  | How often the model ranks the chosen answer above the rejected answer.                     |
| **Rewards/margins**     | Difference between chosen reward and rejected reward; positive is good.                    |
| **Logps/chosen**        | Log probability of the chosen answer; less negative means more likely.                     |
| **Logps/rejected**      | Log probability of the rejected answer; ideally more negative than chosen.                 |


Simple summary: In DPO training, the main goal is to make the model assign higher probability and higher reward to the chosen answer than the rejected answer.


## 11. Save the preference-tuned LoRA adapter and tokenizer


In [23]:
dpo_trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

Preference-tuned LoRA adapter saved to: /content/pharma_tinyllama_preference_dpo_lora_adapter
Saved files:
['adapter_config.json', 'ref', 'adapter_model.safetensors', 'tokenizer_config.json', 'README.md', 'tokenizer.json']


## 12. Push LoRA adapter to Hugging Face Hub

In [24]:
# repo_id = "YOUR_USERNAME/YOUR_REPO_NAME"

In [25]:
# preference_model.push_to_hub(
#     repo_id,
#     private=True,
#     token=WRITE_TOKEN
# )

# tokenizer.push_to_hub(
#     repo_id,
#     private=True,
#     token=WRITE_TOKEN
# )

# print(f"LoRA adapter pushed to: https://huggingface.co/{repo_id}")


## 13. Reload base model + preference adapter for inference


In [26]:
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [27]:
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [28]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [29]:
preference_inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)
preference_inference_model.eval()

print("Base model + preference-tuned LoRA adapter loaded successfully for inference.")

Base model + preference-tuned LoRA adapter loaded successfully for inference.


## 14. Preference-tuned inference helper

We reuse the same `### Instruction: / ### Response:` prompt style used for instruction tuning, since that's typically the format the `prompt` field in the preference dataset follows. Adjust `build_preference_prompt` if your preference data uses a different prompt template.


In [30]:
def build_preference_prompt(instruction, input_text=""):
    instruction = instruction.strip()
    input_text = input_text.strip()

    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )

    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [31]:
def generate_preference_response(instruction, input_text="", max_new_tokens=150):
    prompt = build_preference_prompt(instruction, input_text)

    inputs = inference_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(preference_inference_model.device)

    with torch.no_grad():
        outputs = preference_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

## 15. Test the preference-tuned model


In [32]:
preference_test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why should AI predictions in drug discovery be experimentally validated?",
    "Define pharmacovigilance.",
    "Explain why pharmacovigilance continues after drug approval.",
]

In [33]:
for question in preference_test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_preference_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin has been shown to inhibit glucose-6-phosphatase and to inactivate glucokinase activity, resulting in a reduction in gluconeogenesis. Metformin also inhibits the formation of 1,4-bisphosphoglycerate and glycogen phosphorylase activity. It is also known to have anti-inflammatory effects, as well as being an antioxidant.

### Instruction:
Explain the mechanism of action of aspirin.

### Response:
Aspirin binds to the cyclooxygenase-2 enzyme which results
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
The purpose of the study is to examine whether AI can predict the targeted compounds. The study will utilize a variety of methods that include in vitro cell culture assays, chemical ligand binding, and molecular dynamics simulations to evaluate how well AI models perform at predicting ligands for specific targets in various contexts. In addition, we plan to validate these models using human pharmacokinetics data from the PHARM-HD dataset.

### How will this work be conducted?

#### Research design and methods

#### Outline

  * **Project description**
  * **Research objectives**
  * **Study population**
  * **
QUESTION:
Define pharmacovigilance.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Define pharmacovigilance.

### Response:
The pharmacovigilance of the drug is the process in which the drugs safety and efficacy are monitored during its development, approval and marketing. The pharmacovigilance includes monitoring of the occurrence of adverse events and safety signals for the drugs safety and efficacy. The pharmacovigilance also includes the systematic collection of information on the occurrence of adverse events, safety signals, reports of suspected adverse reactions, and other safety signals from the use of the drug or other medicinal products. The pharmacovigilance also includes the systematic collection of information on the occurrence of adverse events, safety signals, reports of suspected adverse
QUESTION:
Explain why pharmacovigilance continues after drug approval.

MODEL RESPONSE:
### Instruction:
Explain why pharmacovigilance continues after drug approval.

### Response:
Pharmacovigilance is an ongoing process for monitoring the safety of dr

## 16. (Optional) Merge the preference adapter into the base model

This step merges the DPO LoRA adapter weights into the base model weights, producing a standalone preference-tuned model that no longer needs the `peft` library to load.


In [34]:
merge_base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    device_map="auto" if use_cuda else None,
    trust_remote_code=True,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [35]:
# Load the trained preference LoRA adapter on top of the base model.
model_with_preference_adapter = PeftModel.from_pretrained(
    merge_base_model,
    config.adapter_dir
)

In [36]:
# Merge LoRA adapter weights into the base model weights.
merged_preference_model = model_with_preference_adapter.merge_and_unload()

In [37]:
# Save the merged standalone model and tokenizer.
merged_preference_model.save_pretrained(config.merged_model_dir)
inference_tokenizer.save_pretrained(config.merged_model_dir)

print(f"Merged preference-tuned model saved to: {config.merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged preference-tuned model saved to: /content/pharma_tinyllama_preference_merged_model
